In [16]:
import pandas as pd
import numpy as np
import time
import psutil
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report

In [17]:
# Runtime and Memory Tracker
def get_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / (1024 * 1024)  # Convert bytes to MB

start_time = time.time()
mem_before = get_memory_usage()

In [18]:
df = pd.read_csv("Smart Healthcare - Daily Lifestyle Dataset (Wearable device).csv")

In [19]:
df.head()

,ID,Day,Gender,Age (years),Height (meter),Weight (kg),BMI,Step Count,Distance Travel (Km),Blood Pressure,Heart Rate (BPM),Blood Oxygen Level,Sleep Duration (minutes),Screen Time (minute),Earphone Time (minute)
0,1,1,M,27,1.68,60.0,21.3,3255,2.48,120/82,74,99.883661,495,600,63
1,1,2,M,27,1.68,60.0,21.3,3956,3.02,118/83,68,98.661974,405,600,69
2,1,3,M,27,1.68,60.0,21.3,4270,3.26,125/80,73,98.023503,450,690,119
3,1,4,M,27,1.68,60.0,21.3,4562,3.48,120/80,76,97.393352,510,600,50
4,1,5,M,27,1.68,60.0,21.3,6054,4.61,120/80,75,95.519813,489,660,33


### PREPROCESSING

In [20]:
# Handling missing values
df = df.dropna()

# Split Blood Pressure into systolic/diastolic
df[['Systolic', 'Diastolic']] = df['Blood Pressure'].str.split('/', expand=True).astype(float)

# Normalize key columns
scaler = StandardScaler()
numeric_cols = [
    'Age (years)', 'Height (meter)', 'Weight (kg)', 'BMI',
    'Step Count', 'Distance Travel (Km)', 'Heart Rate (BPM)',
    'Blood Oxygen Level', 'Sleep Duration (minutes)',
    'Screen Time (minute)', 'Earphone Time (minute)'
]

# Saving raw copy before normalisation — needed for rule engine
df_raw = df.copy()

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

### FEATURE ENGINEERING

In [21]:
# Activity level
df['Activity_Level'] = df['Step Count'] / 10000

# Sleep quality
df['Sleep_Hours'] = df['Sleep Duration (minutes)'] / 60
df['Sleep_Quality'] = df['Sleep_Hours'].apply(lambda x: 1 if 7 <= x <= 9 else 0)

# Sedentary score
df['Sedentary_Score'] = df['Screen Time (minute)'] + df['Earphone Time (minute)']

# BMI category
def bmi_category(bmi):
    if bmi < -0.5:
        return 0
    elif bmi < 0.5:
        return 1
    else:
        return 2

df['BMI_Category'] = df['BMI'].apply(bmi_category)

In [22]:
df.columns

Index(['ID', 'Day', 'Gender', 'Age (years)', 'Height (meter)', 'Weight (kg)',
       'BMI', 'Step Count', 'Distance Travel (Km)', 'Blood Pressure',
       'Heart Rate (BPM)', 'Blood Oxygen Level', 'Sleep Duration (minutes)',
       'Screen Time (minute)', 'Earphone Time (minute)', 'Systolic',
       'Diastolic', 'Activity_Level', 'Sleep_Hours', 'Sleep_Quality',
       'Sedentary_Score', 'BMI_Category'],
      dtype='object')

### CREATING TARGET VARIABLE

In [23]:
# CREATING TARGET VARIABLE — based on clinical guidelines
# Applied on RAW data before normalisation
# Sources: AHA (2017), WHO Sleep Recommendations, CDC Activity Guidelines

# Pre-parse blood pressure from raw string (e.g. "120/80")
df[['Systolic_raw', 'Diastolic_raw']] = df['Blood Pressure'].str.split('/', expand=True).astype(float)

def create_risk_clinical(row):
    risk_score = 0

    # HEART RATE (AHA Guidelines: normal 60–100 BPM)
    hr = row['Heart Rate (BPM)']
    if hr > 120 or hr < 50:
        risk_score += 2   # Dangerous tachycardia / severe bradycardia
    elif hr > 100 or hr < 60:
        risk_score += 1   # Mild tachycardia / borderline bradycardia

    # BLOOD PRESSURE (AHA 2017 Hypertension Guidelines)
    if row['Systolic_raw'] >= 140 or row['Diastolic_raw'] >= 90:
        risk_score += 2   # Stage 2 Hypertension
    elif row['Systolic_raw'] >= 130 or row['Diastolic_raw'] >= 80:
        risk_score += 1   # Stage 1 Hypertension

    # BLOOD OXYGEN (Clinical hypoxemia thresholds)
    spo2 = row['Blood Oxygen Level']
    if spo2 < 92:
        risk_score += 2   # Clinically significant hypoxemia
    elif spo2 < 95:
        risk_score += 1   # Mild hypoxemia

    # SLEEP DURATION (WHO/NSF: 7–9 hrs for adults = 420–540 min)
    sleep_hours = row['Sleep Duration (minutes)'] / 60
    if sleep_hours < 5:
        risk_score += 2   # Severe sleep deprivation
    elif sleep_hours < 7:
        risk_score += 1   # Below recommended range

    # PHYSICAL ACTIVITY (CDC: 7,000–10,000 steps/day)
    if row['Step Count'] < 3000:
        risk_score += 2   # Severely sedentary
    elif row['Step Count'] < 7000:
        risk_score += 1   # Below recommended activity

    # BMI (WHO Classification)
    if row['BMI'] >= 30 or row['BMI'] < 18.5:
        risk_score += 1   # Obese or underweight

    # Map cumulative score to risk level
    if risk_score >= 4:
        return 2   # High Risk
    elif risk_score >= 2:
        return 1   # Medium Risk
    else:
        return 0   # Low Risk

df['Risk_Level'] = df.apply(create_risk_clinical, axis=1)

### MODEL TRAINING

In [24]:
features = [
    'Age (years)', 'BMI', 'Step Count', 'Heart Rate (BPM)',
    'Blood Oxygen Level', 'Sleep Duration (minutes)',
    'Sedentary_Score', 'Activity_Level', 'Systolic', 'Diastolic'
]

X = df[features]
y = df['Risk_Level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           2       1.00      1.00      1.00       140

    accuracy                           1.00       140
   macro avg       1.00      1.00      1.00       140
weighted avg       1.00      1.00      1.00       140



### ANOMALY DETECTION

In [25]:
iso = IsolationForest(contamination=0.05)
df['Anomaly'] = iso.fit_predict(X)

### REASONING ENGINE

In [26]:
# REASONING ENGINE — clinically grounded thresholds
# Works on RAW (unscaled) values
# Sources: AHA (2017), WHO, CDC, NSF

def rule_engine(raw_row):
    alerts = []

    # Parse blood pressure
    systolic, diastolic = map(float, str(raw_row['Blood Pressure']).split('/'))
    sleep_hours = raw_row['Sleep Duration (minutes)'] / 60

    # --- HEART RATE (AHA: normal 60–100 BPM) ---
    hr = raw_row['Heart Rate (BPM)']
    if hr > 120:
        alerts.append(f"Dangerous tachycardia: HR = {hr} BPM (AHA threshold: >120)")
    elif hr > 100:
        alerts.append(f"Elevated heart rate: HR = {hr} BPM (AHA normal: 60–100)")
    elif hr < 50:
        alerts.append(f"Severe bradycardia: HR = {hr} BPM (AHA threshold: <50)")
    elif hr < 60:
        alerts.append(f"Low heart rate: HR = {hr} BPM (AHA normal: 60–100)")

    # --- BLOOD PRESSURE (AHA 2017) ---
    if systolic >= 140 or diastolic >= 90:
        alerts.append(f"Stage 2 Hypertension: {int(systolic)}/{int(diastolic)} mmHg (AHA 2017: ≥140/90)")
    elif systolic >= 130 or diastolic >= 80:
        alerts.append(f"Stage 1 Hypertension: {int(systolic)}/{int(diastolic)} mmHg (AHA 2017: ≥130/80)")

    # --- BLOOD OXYGEN (Clinical thresholds) ---
    spo2 = raw_row['Blood Oxygen Level']
    if spo2 < 92:
        alerts.append(f"Hypoxemia detected: SpO2 = {spo2:.1f}% (clinical threshold: <92%)")
    elif spo2 < 95:
        alerts.append(f"Mildly low oxygen: SpO2 = {spo2:.1f}% (normal: ≥95%)")

    # --- SLEEP (WHO/NSF: 7–9 hrs for adults) ---
    if sleep_hours < 5:
        alerts.append(f"Severe sleep deprivation: {sleep_hours:.1f} hrs (WHO minimum: 7 hrs)")
    elif sleep_hours < 7:
        alerts.append(f"Insufficient sleep: {sleep_hours:.1f} hrs (WHO recommendation: 7–9 hrs)")

    # --- PHYSICAL ACTIVITY (CDC: 7,000–10,000 steps/day) ---
    steps = raw_row['Step Count']
    if steps < 3000:
        alerts.append(f"Severely sedentary: {steps} steps/day (CDC recommendation: 7,000+)")
    elif steps < 7000:
        alerts.append(f"Low activity: {steps} steps/day (below CDC recommendation of 7,000)")

    return alerts

### AI AGENT FUNCTION

In [27]:
def health_agent(input_row):
    row_df = pd.DataFrame([input_row])

    # Scale input
    row_df[numeric_cols] = scaler.transform(row_df[numeric_cols])

    # Feature engineering (same as above)
    row_df['Activity_Level'] = row_df['Step Count'] / 10000
    row_df['Sleep_Hours'] = row_df['Sleep Duration (minutes)'] / 60
    row_df['Sleep_Quality'] = row_df['Sleep_Hours'].apply(lambda x: 1 if 7 <= x <= 9 else 0)
    row_df['Sedentary_Score'] = row_df['Screen Time (minute)'] + row_df['Earphone Time (minute)']

    # Split BP
    row_df[['Systolic', 'Diastolic']] = row_df['Blood Pressure'].str.split('/', expand=True).astype(float)

    # Prediction
    risk = model.predict(row_df[features])[0]

    # Anomaly
    anomaly = iso.predict(row_df[features])[0]

    # Rules
    rules = rule_engine(input_row)

    # Output
    risk_map = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}

    output = {
        "Alert Level": risk_map[risk],
        "Anomaly Detected": True if anomaly == -1 else False,
        "Reasons": rules,
        "Recommendation": []
    }

    # Recommendations — updated to match new rule engine alert strings
    if any("Low activity" in r or "Severely sedentary" in r for r in rules):
        output["Recommendation"].append("Increase daily steps to 7,000+ (CDC recommendation)")

    if any("sleep" in r.lower() for r in rules):
        output["Recommendation"].append("Aim for 7–9 hours of sleep (WHO recommendation)")

    if any("Hypertension" in r for r in rules):
        output["Recommendation"].append("Monitor blood pressure and consult a healthcare provider")

    if any("heart rate" in r.lower() for r in rules):
        output["Recommendation"].append("Monitor heart rate and reduce physical/mental stress")

    if any("oxygen" in r.lower() or "Hypoxemia" in r for r in rules):
        output["Recommendation"].append("Check oxygen levels and seek medical advice if persistent")

    return output

### TEST

In [28]:
sample = df_raw.iloc[0].to_dict()
result = health_agent(sample)

print("\nAI AGENT OUTPUT:\n")
for k, v in result.items():
    print(f"{k}: {v}")


AI AGENT OUTPUT:

Alert Level: HIGH
Anomaly Detected: False
Reasons: ['Stage 1 Hypertension: 120/82 mmHg (AHA 2017: ≥130/80)', 'Low activity: 3255 steps/day (below CDC recommendation of 7,000)']
Recommendation: ['Increase daily steps to 7,000+ (CDC recommendation)', 'Monitor blood pressure and consult a healthcare provider']


In [29]:
end_time = time.time()
total_runtime = end_time - start_time
print(f"Total Runtime: {total_runtime:.4f} seconds")

mem_after = get_memory_usage()
print(f"Memory Used: {mem_after - mem_before:.2f} MB")
print(f"Peak RAM: {mem_after:.2f} MB")

Total Runtime: 0.7033 seconds
Memory Used: 15.35 MB
Peak RAM: 96.19 MB
